# Engenharia De Modelos

## Objetivos da engenharia de modelos para previsão de churns

A engenharia de modelos para previsão de churn tem como objetivo testar e otimizar o modelo através de testes de diferentes técnicas. Serão aplicados:

- Técnicas de Feature Engineering
- Pipelines de processamento para automação
- Tratamento de dados desbalanceados
- Otimizar hiperparâmetros
- Validação de modelos com técnicas apropriadas

# 1 - Configuração de ambiente e set up de bilbiotecas


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.experimental import enable_halving_search_cv  # noqa: F401
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, cross_val_score, RepeatedStratifiedKFold, HalvingRandomSearchCV
from sklearn.preprocessing import StandardScaler, RobustScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier, ExtraTreesClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve, accuracy_score, precision_score, recall_score, f1_score
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif
from scipy.stats import randint, uniform
import mlflow
import mlflow.sklearn
import joblib
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')

## 2. Carregamento Dataset Pré-processado na [Aula 2](../aula_02_experimentacao_mvp/)

In [4]:
# Carregar dataset pré-processado da Aula 2
path = "../data/pre-processed/Telco_customer_churn_preprocessed.csv"

df = pd.read_csv(path)

# Garantir coluna alvo presente e binária (já tratada na Aula 2)
assert 'target' in df.columns, "Coluna 'target' não encontrada no dataset pré-processado."

# Remover colunas de metadados se existirem
for col in ['id', 'dataset']:
    if col in df.columns:
        df.drop(columns=[col], inplace=True)

print(f"Dataset shape (pré-processado): {df.shape}")
print("\nTarget (0=Sem churn, 1=Com churn):")
print(df['target'].value_counts())

df.head()

Dataset shape (pré-processado): (7043, 58)

Target (0=Sem churn, 1=Com churn):
target
0    5174
1    1869
Name: count, dtype: int64


,CustomerID,Count,Zip Code,Latitude,Longitude,Tenure Months,Monthly Charges,Total Charges,target,Churn Score,...,Churn Reason_Lack of self-service on Website,Churn Reason_Limited range of services,Churn Reason_Long distance charges,Churn Reason_Moved,Churn Reason_Network reliability,Churn Reason_Poor expertise of online support,Churn Reason_Poor expertise of phone support,Churn Reason_Price too high,Churn Reason_Product dissatisfaction,Churn Reason_Service dissatisfaction
0,3668-QPYBK,1,90003,33.964131,-118.272783,2,53.85,108.15,1,86,...,False,False,False,False,False,False,False,False,False,False
1,9237-HQITU,1,90005,34.059281,-118.307420,2,70.70,151.65,1,67,...,False,False,False,True,False,False,False,False,False,False
2,9305-CDSKC,1,90006,34.048013,-118.293953,8,99.65,820.50,1,86,...,False,False,False,True,False,False,False,False,False,False
3,7892-POOKP,1,90010,34.062125,-118.315709,28,104.80,3046.05,1,84,...,False,False,False,True,False,False,False,False,False,False
4,0280-XJGEX,1,90015,34.039224,-118.266293,49,103.70,5036.30,1,89,...,False,False,False,False,False,False,False,False,False,False


In [5]:
# Estatísticas rápidas para conferência (dataset já pré-processado)
df.describe()

,Count,Zip Code,Latitude,Longitude,Tenure Months,Monthly Charges,Total Charges,target,Churn Score,CLTV
count,7043.0,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000
mean,1.0,93521.964646,36.282441,-119.798880,32.371149,64.761692,2279.734304,0.265370,58.699418,4400.295755
std,0.0,1865.794555,2.455723,2.157889,24.559481,30.090047,2266.794470,0.441561,21.525131,1183.057152
min,1.0,90001.000000,32.555828,-124.301372,0.000000,18.250000,0.000000,0.000000,5.000000,2003.000000
25%,1.0,92102.000000,34.030915,-121.815412,9.000000,35.500000,398.550000,0.000000,40.000000,3469.000000
50%,1.0,93552.000000,36.391777,-119.730885,29.000000,70.350000,1394.550000,0.000000,61.000000,4527.000000
75%,1.0,95351.000000,38.224869,-118.043237,55.000000,89.850000,3786.600000,1.000000,75.000000,5380.500000
max,1.0,96161.000000,41.962127,-114.192901,72.000000,118.750000,8684.800000,1.000000,100.000000,6500.000000


In [6]:
# Verificar (rapidamente) valores nulos - não deve haver após Aula 2
null_total = df.isnull().sum().sum()
print(f"Total de valores nulos no CSV pré-processado: {null_total}")
if null_total > 0:
    print(df.isnull().sum().sort_values(ascending=False).head(10))

Total de valores nulos no CSV pré-processado: 0


## 3. Feature Engineering

### Tarefa 1: Criar novas features baseadas nas existentes